In [8]:
import pandas as pd
import numpy as np

df={
    'customers': pd.read_csv('../../Olist_Data/olist_customers_dataset.csv'),
    'orders': pd.read_csv('../../Olist_Data/olist_orders_dataset.csv'),
    'order_items': pd.read_csv('../../Olist_Data/olist_order_items_dataset.csv'),
    'payments': pd.read_csv('../../Olist_Data/olist_order_payments_dataset.csv'),
    'reviews': pd.read_csv('../../Olist_Data/olist_order_reviews_dataset.csv'),
    'products': pd.read_csv('../../Olist_Data/olist_products_dataset.csv'),
    'sellers': pd.read_csv('../../Olist_Data/olist_sellers_dataset.csv'),
    'geolocation': pd.read_csv('../../Olist_Data/olist_geolocation_dataset.csv')
}

for name,table in df.items():
    duplicate_count=table.duplicated().sum()
    print(f"Table {name}: {duplicate_count} duplicate rows")

Table customers: 0 duplicate rows
Table orders: 0 duplicate rows
Table order_items: 0 duplicate rows
Table payments: 0 duplicate rows
Table reviews: 0 duplicate rows
Table products: 0 duplicate rows
Table sellers: 0 duplicate rows
Table geolocation: 261831 duplicate rows


In [9]:
df['geolocation'][df['geolocation'].duplicated(keep=False)].head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [10]:
before = df['geolocation'].shape[0]
df['geolocation'] = df['geolocation'].drop_duplicates()
after = df['geolocation'].shape[0]
print(f"geolocation: {before} rows before, {after} rows after, removed {before - after}")

geolocation: 1000163 rows before, 738332 rows after, removed 261831


In [11]:
df['geolocation']['geolocation_city'].nunique()
df['geolocation']['geolocation_city'].unique()[:30]

<ArrowStringArray>
[             'sao paulo',              'são paulo',  'sao bernardo do campo',
                'jundiaí',        'taboão da serra',               'sãopaulo',
                     'sp',             'sa£o paulo',    'sao jose dos campos',
                 'osasco',            'carapicuíba',            'carapicuiba',
                'barueri',    'santana de parnaiba',  'pirapora do bom jesus',
    'santana de parnaíba',                'jandira',                'itapevi',
                  'cotia',        'taboao da serra', 'vargem grande paulista',
         'embu das artes',   'itapecerica da serra',                   'embu',
  'são lourenço da serra',  'sao lourenco da serra',             'embu-guacu',
             'embu-guaçu',             'embu guaçu',              'juquitiba']
Length: 30, dtype: str

In [12]:
df['geolocation']['geolocation_city'] = df['geolocation']['geolocation_city'].str.lower().str.strip()
df['geolocation']['geolocation_city'].nunique()

8010

In [13]:
from rapidfuzz import process, fuzz

unique_cities = df['geolocation']['geolocation_city'].unique()
len(unique_cities)

8010

In [14]:
# find the most frequent city name for each zip code
most_common_city = df['geolocation'].groupby('geolocation_zip_code_prefix')['geolocation_city'].agg(lambda x: x.mode()[0])

# map that back onto every row
df['geolocation']['most_common_city_for_zip'] = df['geolocation']['geolocation_zip_code_prefix'].map(most_common_city)

# count how many rows have a city name that disagrees with the most common one for that zip
mismatches = (df['geolocation']['geolocation_city'] != df['geolocation']['most_common_city_for_zip']).sum()
print(f"{mismatches} rows have a city name that differs from the most common name for their zip code")

65163 rows have a city name that differs from the most common name for their zip code


In [15]:
mismatch_rows = df['geolocation'][df['geolocation']['geolocation_city'] != df['geolocation']['most_common_city_for_zip']]
mismatch_rows[['geolocation_zip_code_prefix', 'geolocation_city', 'most_common_city_for_zip']].head(20)

,geolocation_zip_code_prefix,geolocation_city,most_common_city_for_zip
5,1012,são paulo,sao paulo
14,1037,são paulo,sao paulo
17,1024,são paulo,sao paulo
21,1020,são paulo,sao paulo
22,1011,são paulo,sao paulo
23,1043,são paulo,sao paulo
28,1032,são paulo,sao paulo
31,1037,são paulo,sao paulo
32,1017,são paulo,sao paulo
57,1046,são paulo,sao paulo


In [16]:
canonical_cities = most_common_city.unique()
len(canonical_cities)

5832

In [17]:
from rapidfuzz import process, fuzz

sample_cities = unique_cities[:10]

for city in sample_cities:
    match, score, idx = process.extractOne(city, canonical_cities, scorer=fuzz.ratio)
    print(f"{city!r} -> best match: {match!r} (score: {score:.1f})")

'sao paulo' -> best match: 'sao paulo' (score: 100.0)
'são paulo' -> best match: 'são paulo' (score: 100.0)
'sao bernardo do campo' -> best match: 'sao bernardo do campo' (score: 100.0)
'jundiaí' -> best match: 'jundia' (score: 92.3)
'taboão da serra' -> best match: 'taboão da serra' (score: 100.0)
'sãopaulo' -> best match: 'são paulo' (score: 94.1)
'sp' -> best match: 'sape' (score: 66.7)
'sa£o paulo' -> best match: 'sao paulo' (score: 94.7)
'sao jose dos campos' -> best match: 'sao jose dos campos' (score: 100.0)
'osasco' -> best match: 'osasco' (score: 100.0)


In [18]:
replacements = {}

for city in unique_cities:
    match, score, idx = process.extractOne(city, canonical_cities, scorer=fuzz.ratio)
    if score >= 90 and score < 100:
        replacements[city] = match

print(f"Found {len(replacements)} names to fix")
# look at a sample before applying
list(replacements.items())[:15]

Found 1160 names to fix


[('jundiaí', 'jundia'),
 ('sãopaulo', 'são paulo'),
 ('sa£o paulo', 'sao paulo'),
 ('santana de parnaíba', 'santana de parnaiba'),
 ('são lourenço da serra', 'sao lourenco da serra'),
 ('embu-guaçu', 'embu-guacu'),
 ('embu guacu', 'embu-guacu'),
 ('embuguacu', 'embu-guacu'),
 ('jordanésia', 'jordanesia'),
 ('mogidascruzes', 'mogi das cruzes'),
 ('salesópolis', 'salesopolis'),
 ('biritiba mirim', 'biritiba-mirim'),
 ('ribeirão pires', 'ribeirao pires'),
 ('são caetano do sul', 'sao caetano do sul'),
 ('pariquera-açu', 'pariquera-acu')]

In [19]:
df['geolocation']['geolocation_city'] = df['geolocation']['geolocation_city'].replace(replacements)
df['geolocation']['geolocation_city'].nunique()


6850

In [20]:
for name, table in df.items():
    print(f"\n{name}:")
    print(table.dtypes)


customers:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

orders:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

order_items:
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

payments:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

reviews:
review_id                    str
order_id    

In [21]:
df['orders']['order_purchase_timestamp'] = pd.to_datetime(df['orders']['order_purchase_timestamp'])
df['orders']['order_approved_at'] = pd.to_datetime(df['orders']['order_approved_at'])
df['orders']['order_delivered_carrier_date'] = pd.to_datetime(df['orders']['order_delivered_carrier_date'])
df['orders']['order_delivered_customer_date'] = pd.to_datetime(df['orders']['order_delivered_customer_date'])
df['orders']['order_estimated_delivery_date'] = pd.to_datetime(df['orders']['order_estimated_delivery_date'])

df['order_items']['shipping_limit_date'] = pd.to_datetime(df['order_items']['shipping_limit_date'])

df['reviews']['review_creation_date'] = pd.to_datetime(df['reviews']['review_creation_date'])
df['reviews']['review_answer_timestamp'] = pd.to_datetime(df['reviews']['review_answer_timestamp'])

print("Dates converted")

Dates converted


In [22]:
df['orders'].dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [23]:
df['orders']['order_status'] = df['orders']['order_status'].astype('category')
df['payments']['payment_type'] = df['payments']['payment_type'].astype('category')
df['products']['product_category_name'] = df['products']['product_category_name'].astype('category')
df['customers']['customer_state'] = df['customers']['customer_state'].astype('category')
df['sellers']['seller_state'] = df['sellers']['seller_state'].astype('category')
df['geolocation']['geolocation_state'] = df['geolocation']['geolocation_state'].astype('category')

print("Category dtypes applied")

Category dtypes applied


In [24]:
df['orders']['order_status'].dtype

CategoricalDtype(categories=['approved', 'canceled', 'created', 'delivered', 'invoiced',
                  'processing', 'shipped', 'unavailable'],
, ordered=False, categories_dtype=str)